# Wang zero-frequency validation on a surface-piercing ellipsoid

For an ellipsoid with semi-axes $(a,b,c)$, the classical potential coefficients are

$$\alpha_i=abc\int_0^\infty\frac{d\lambda}{(a_i^2+\lambda)\sqrt{(a^2+\lambda)(b^2+\lambda)(c^2+\lambda)}},$$

and the full-body translational added masses are

$$m_i=\rho\frac{4\pi abc}{3}\frac{\alpha_i}{2-\alpha_i}.$$

The waterplane is a symmetry plane, so the surface-piercing lower half has one-half of the full-body sway added mass. In Wang's force convention the expected derivative is $Y_{\dot v}=-m_y/2$.

In [ ]:
using Revise
using Pkg

function marinehydro_root(start=pwd())
    directory = abspath(start)
    while true
        project = joinpath(directory, "Project.toml")
        if isfile(project) && occursin("name = \"MarineHydro\"", read(project, String))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Run Jupyter from the MarineHydro.jl repository.")
        directory = parent
    end
end

project_root = marinehydro_root()
Pkg.activate(project_root)
using MarineHydro
using CairoMakie
CairoMakie.activate!()

In [ ]:
semi_axes = (3.0, 1.0, 0.8)
density = 1000.0
forward_speed = 1.0
shapes = [(8, 4), (12, 6), (16, 8), (20, 10)]
frequencies = [0.2, 0.1, 0.05, 0.02, 0.01]

coefficients = ellipsoid_potential_coefficients(semi_axes)
analytical_full_body = ellipsoid_added_mass(semi_axes; density=density)
analytical_sway = analytical_full_body.y / 2
analytical_volume = 2pi * prod(semi_axes) / 3
(; coefficients, coefficient_sum=sum(coefficients), analytical_sway, analytical_volume)

## Mesh refinement

The analytical geometry and potential solution are checked separately. Faceted volume converges monotonically from below, while the constant-panel BEM sway added mass approaches $m_y/2$ from above for this mesh family.

In [ ]:
mesh_rows = map(shapes) do shape
    mesh = surface_piercing_ellipsoid_mesh(
        semi_axes; longitudinal_panels=shape[1], girth_panels=shape[2],
    )
    result = solve_wang_maneuvering(mesh, forward_speed; rho=density)
    volume = mesh_signed_volume(mesh)
    added_mass = -result.derivatives.Y_vdot
    (; shape, panels=mesh.nfaces, volume, added_mass,
       volume_relative_error=volume / analytical_volume - 1,
       added_mass_relative_error=added_mass / analytical_sway - 1,
       boundary_residual=result.boundary_residual,
       reciprocity_error=result.derivatives.N_vdot - result.derivatives.Y_rdot)
end
mesh_rows

## Independent low-frequency limit

MarineHydro's ordinary radiation calculation is independent of the Wang extraction. Its sway added mass should approach the same zero-frequency limit as $\omega\to0$, with damping tending to zero.

In [ ]:
radiation_shape = (16, 8)
radiation_mesh = surface_piercing_ellipsoid_mesh(
    semi_axes; longitudinal_panels=radiation_shape[1], girth_panels=radiation_shape[2],
)
wang_zero_frequency = -solve_wang_maneuvering(
    radiation_mesh, forward_speed; rho=density,
).derivatives.Y_vdot
previous_density = SETTINGS.rho
radiation_rows = try
    set_rho!(density)
    map(frequencies) do omega
        added_mass, damping = calculate_radiation_forces(
            radiation_mesh, [0.0, 1.0, 0.0], omega,
        )
        (; omega, added_mass, damping,
           analytical_relative_error=added_mass / analytical_sway - 1,
           wang_relative_difference=added_mass / wang_zero_frequency - 1)
    end
finally
    set_rho!(previous_density)
end
radiation_rows

In [ ]:
figure = Figure(size=(1350, 560), backgroundcolor=:white)
error_axis = Axis(figure[1, 1]; title="Surface-piercing ellipsoid mesh convergence",
    xlabel="quadrilateral panels", ylabel="absolute relative error",
    xscale=log10, yscale=log10)
panels = [row.panels for row in mesh_rows]
volume_error = abs.([row.volume_relative_error for row in mesh_rows])
added_mass_error = abs.([row.added_mass_relative_error for row in mesh_rows])
scatterlines!(error_axis, panels, volume_error; color=:dodgerblue3, linewidth=2.5,
    marker=:circle, markersize=10, label="faceted volume")
scatterlines!(error_axis, panels, added_mass_error; color=:orangered2, linewidth=2.5,
    marker=:rect, markersize=10, label="Wang sway added mass")
axislegend(error_axis; position=:rt)

frequency_axis = Axis(figure[1, 2]; title="Low-frequency sway added mass",
    xlabel="ω [rad/s]", ylabel="added mass [kg]", xscale=log10)
omega = [row.omega for row in radiation_rows]
radiation_mass = [row.added_mass for row in radiation_rows]
scatterlines!(frequency_axis, omega, radiation_mass; color=:purple3, linewidth=2.5,
    marker=:circle, markersize=10, label="radiation BEM")
hlines!(frequency_axis, [wang_zero_frequency]; color=:orangered2, linewidth=2,
    linestyle=:dash, label="Wang, same mesh")
hlines!(frequency_axis, [analytical_sway]; color=:black, linewidth=2,
    linestyle=:dot, label="analytical ellipsoid")
axislegend(frequency_axis; position=:rb)

Label(figure[0, :], "Independent checks of Wang's zero-frequency acceleration derivative";
    fontsize=22, font=:bold)
output_directory = joinpath(project_root, "validation", "wang2000", "results")
mkpath(output_directory)
output_path = joinpath(output_directory, "ellipsoid_validation.png")
save(output_path, figure; px_per_unit=1.5)
figure

## Interpretation

For $(a,b,c)=(3,1,0.8)$ m and $\rho=1000$ kg/m³, the analytical lower-half sway added mass is approximately $3340.62$ kg. The $16\times8$ radiation mesh is already within about $0.6\%$ of that value at $\omega=0.01$ rad/s. The constant-panel Wang sequence converges more slowly and from the opposite side, which makes this a useful discretization check rather than a tuned agreement.